## 主題：
AI Agents-打造你專屬的超級代理人【Demo07a】
《預言家日報》記者的AI超級助手
- 任務目標：
打造一個幫助記者完成「新聞撰寫」與「社群貼文優化」的 AI Agent，採用 Reflection 設計模式，具備 Planning 思考能力，並以 Gradio 展示互動介面。
## 🧠 設計概念
- 背景設定：哈利波特魔法世界的記者（使用者）

- 所屬媒體：《預言家日報》（The Daily Prophet）

## 🧙‍♂️Agent 功能任務：

✅ 幫忙生成新聞草稿（含標題與導語）

✅ 分析草稿，提出優化建議

✅ 進行反思（Reflection）並自動修正草稿

✅ 擷取草稿重點，轉為社群貼文（Twitter、Threads、IG等）

✅ 提供不同語氣風格（嚴肅新聞 / 魔法八卦 / 偵探推理風）

## 實作技術路線：

🎯 使用 OpenAI / Groq 的 LLM 作為主要語言模型

🔁 使用 Reflection 模式：一個 LLM 生成、一個 LLM 評估（或同一模型扮演兩角）

⚙️ Gradio 打造互動介面（輸入草稿、看建議、生成優化版本與社群文）

## 🧩 心得反思

透過本次實作，我了解到 **AI Agents 的價值不只在於一次性生成，而在於具備反思與修正的能力（Reflection）**，
也讓我更了解 AI 是「智慧寫作夥伴」而非單一工具，能夠更有效地產出高品質內容!
我這次使用 Groq 提供的 LLaMA3 模型，在效能與語言理解上也表現出色，最重要的是，它是免費的哈哈哈~~
此外，使用 Gradio 架設互動介面，讓整個使用流程更加直觀與親切，我認為這是日後開發個人化 AI 應用的關鍵 DEMO 之一。
以下是我這次作業的內容


In [1]:
!pip install openai gradio --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.8 MB/s eta 0:00:00


In [5]:
# 📌 第二步：引入必要套件
import os
from openai import OpenAI
import gradio as gr
from google.colab import userdata

#  設定 Groq API
api_key = "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"
os.environ['GROQ_API_KEY'] = api_key

#  初始化 Groq Client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key
)

model = "llama3-70b-8192"

In [10]:
# 📌 第三步：設計 Reflection 模式的邏輯
# ----------------------
# 模擬兩個角色：作者（Author LLM）與 評論員（Reviewer LLM）

def generate_draft(prompt):
    messages = [
        {"role": "system", "content": "你是哈利波特魔法世界的記者，使用繁體中文撰寫《預言家日報》的新聞草稿，包括新聞標題與導語段落，使用繁體中文，約200字。"},
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content.strip()

def review_draft(draft):
    messages = [
        {"role": "system", "content": "你是一位資深新聞評論員，使用繁體中文幫忙檢視以下草稿並提供具體優化建議。"},
        {"role": "user", "content": draft}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content.strip()

def revise_draft(draft, feedback):
    messages = [
        {"role": "system", "content": "你是一位新聞記者，使用繁體中文根據以下建議，幫忙修正新聞草稿。"},
        {"role": "user", "content": f"草稿如下：\n{draft}\n\n建議如下：\n{feedback}"}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content.strip()

In [12]:
# 📌 第四步：延伸功能 — 生成社群發文版本
def generate_social_post(news):
    messages = [
        {"role": "system", "content": "你是社群小編，使用繁體中文將下列新聞內容轉換為一則適合貼在 IG/Twitter 的貼文，語氣可以活潑一點，並加入適當 hashtag。"},
        {"role": "user", "content": news}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content.strip()

In [13]:
# 📌 第五步：用 Gradio 建立互動介面
with gr.Blocks(title="預言家日報 AI 超級助手") as demo:
    gr.Markdown("## 📰 預言家日報記者小幫手")
    gr.Markdown("請輸入你想報導的魔法世界事件，AI 會幫你撰寫草稿、提供建議並優化新聞，同時產出社群貼文版本。")

    user_input = gr.Textbox(label="輸入新聞主題（Prompt）", placeholder="例：霍格華茲食堂驚現飛天饅頭事件", lines=2)
    gen_btn = gr.Button("✨ 生成新聞草稿")

    draft_output = gr.Textbox(label="📝 新聞草稿")
    review_output = gr.Textbox(label="📋 評論與建議")
    revised_output = gr.Textbox(label="🪄 優化後新聞")
    social_output = gr.Textbox(label="📣 社群貼文版本")

    def run_agent(prompt):
        draft = generate_draft(prompt)
        review = review_draft(draft)
        revised = revise_draft(draft, review)
        social = generate_social_post(revised)
        return draft, review, revised, social

    gen_btn.click(run_agent, inputs=[user_input], outputs=[draft_output, review_output, revised_output, social_output])

demo.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://75f155e2044f1f5d95.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
